# 🔱 VoiceBatch Studio v2.6.0 - [IndicF5 Edition]
IIT Madras का शक्तिशाली हिंदी क्लोनिंग इंजन।

In [ ]:
# @title 💤 Step 1: IndicF5 इंस्टॉलेशन
import os
from IPython.display import display, Javascript
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ IndicF5 लोड हो रहा है (पहली बार में 2-4 मिनट लग सकते हैं)...")
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q gradio librosa soundfile torchaudio torchcodec
os.makedirs("outputs", exist_ok=True)
print("✅ IndicF5 तैयार है!")

In [ ]:
# @title 🚀 Step 2: ऐप लॉन्च करें (IndicF5 Cloning)
app_code = r'''
import gradio as gr
import torch, librosa, os, re, numpy as np, soundfile as sf
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import load_model, infer_process

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# मॉडल लोड करना (IndicF5 Official)
checkpoint_path = "ai4bharat/IndicF5"
model_obj = load_model(DiT, checkpoint_path, device)

def indic_engine(gen_text, ref_audio, ref_text, speed):
    if not ref_audio: return None
    
    final_path = 'outputs/IndicF5_Output.wav'
    
    # IndicF5 Inference
    # नोट: इसमें ref_text देना जरूरी है जो ऑडियो में बोला गया है
    audio, sr = infer_process(ref_audio, ref_text, gen_text, model_obj, device, speed=speed)
    
    sf.write(final_path, audio, sr)
    return final_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="green")) as demo:
    gr.Markdown('# 🎙️ IndicF5 Studio v2.6.0')
    with gr.Row():
        with gr.Column():
            gen_txt = gr.Textbox(label='जो बुलवाना है (Hindi Script)', lines=8)
            ref_aud = gr.Audio(label='Voice Sample (10 Sec)', type='filepath')
            ref_txt = gr.Textbox(label='Sample में क्या बोला गया है? (Reference Text)', placeholder='सैंपल की आवाज़ का टेक्स्ट यहाँ लिखें...')
            spd = gr.Slider(0.7, 1.4, 1.0, label="Speed")
            btn = gr.Button('Clone & Generate ⚡', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Output Audio')
            gr.Markdown('### Tips:\n1. Sample कम से कम 10 सेकंड का रखें।\n2. Reference Text सटीक लिखें ताकि क्लोनिंग परफेक्ट हो।')

    btn.click(indic_engine, [gen_txt, ref_aud, ref_txt, spd], out)
demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
!python app.py